In [14]:
# Data Manipulation
import pandas as pd
import numpy as np
import pandas_ta as ta

# Data Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Gradient Boosting
from xgboost import XGBClassifier

# Model Validation
from sklearn import metrics

# Warnings
import warnings
warnings.filterwarnings('ignore')

NumPy: 2.2.6
Pandas: 3.0.3
Scikit-learn: 1.9.0
XGBoost: 3.3.0
pandas-ta: <module 'pandas_ta' from 'C:\\Users\\gab\\anaconda3\\envs\\bitcoin_ml\\Lib\\site-packages\\pandas_ta\\__init__.py'>


In [2]:
btc=pd.read_csv('bitcoin.csv')
btc.head()
btc.shape
btc.info()
btc.describe()
btc['Date']=pd.to_datetime(btc['Date'])
btc.drop('Adj Close',axis=1,inplace=True)

In [ ]:
# Check for duplicate rows
btc.duplicated().sum()

# Check for missing values
btc.isnull().sum()

# Number of unique values per column
btc.nunique()

In [3]:
btc['Target']=np.where(btc['Close'].shift(-1)>btc['Close'],1,0)
btc['range']=btc['High']-btc['Low']
btc['EMA5'] = ta.ema(btc['Close'], length=5)
btc['EMA10'] = ta.ema(btc['Close'], length=10)
btc['EMA20'] = ta.ema(btc['Close'], length=20)
btc['EMA50'] = ta.ema(btc['Close'], length=50)
btc['EMA100'] = ta.ema(btc['Close'], length=100)
btc['EMA200'] = ta.ema(btc['Close'], length=200)
btc['SMA10'] = ta.sma(btc['Close'], length=10)
btc['SMA20'] = ta.sma(btc['Close'], length=20)
btc['SMA50'] = ta.sma(btc['Close'], length=50)
btc['SMA100'] = ta.sma(btc['Close'], length=100)
btc['SMA200'] = ta.sma(btc['Close'], length=200)
btc['WMA20'] = ta.wma(btc['Close'], length=20)
btc['RSI14'] = ta.rsi(btc['Close'], length=14)
btc['HMA20'] = ta.hma(btc['Close'], length=20)
btc['ROC10'] = ta.roc(btc['Close'], length=10)
btc['MOM10'] = ta.mom(btc['Close'], length=10)
btc['CCI20'] = ta.cci(
    btc['High'],
    btc['Low'],
    btc['Close'],
    length=20
)
stoch = ta.stoch(
    btc['High'],
    btc['Low'],
    btc['Close']
)
btc['WilliamsR'] = ta.willr(
    btc['High'],
    btc['Low'],
    btc['Close']
)
btc = pd.concat([btc, stoch], axis=1)
btc['RSI'] = ta.rsi(btc['Close'], length=14)

macd = ta.macd(btc['Close'])
btc = pd.concat([btc, macd], axis=1)

bbands = ta.bbands(btc['Close'])
btc = pd.concat([btc, bbands], axis=1)

btc['ATR14'] = ta.atr(
    btc['High'],
    btc['Low'],
    btc['Close'],
    length=14
)
dc = ta.donchian(
    btc['High'],
    btc['Low']
)
btc = pd.concat([btc, dc], axis=1)
kc = ta.kc(
    btc['High'],
    btc['Low'],
    btc['Close']
)
btc = pd.concat([btc, kc], axis=1)
btc['OBV'] = ta.obv(
    btc['Close'],
    btc['Volume']
)
btc['CMF'] = ta.cmf(
    btc['High'],
    btc['Low'],
    btc['Close'],
    btc['Volume']
)
btc['MFI'] = ta.mfi(
    btc['High'],
    btc['Low'],
    btc['Close'],
    btc['Volume']
)
btc['AD'] = ta.ad(
    btc['High'],
    btc['Low'],
    btc['Close'],
    btc['Volume']
)
btc['Body'] = btc['Close'] - btc['Open']
btc['BodyPct'] = btc['Body'] / btc['range']
btc['UpperShadow'] = btc['High'] - btc[['Open','Close']].max(axis=1)
btc['UpperShadow'] = btc['High'] - btc[['Open','Close']].max(axis=1)
btc = pd.concat([btc, kc], axis=1)
btc = pd.concat([btc, dc], axis=1)
btc['MA20'] = btc['Close'].rolling(20).mean()
btc['STD20'] = btc['Close'].rolling(20).std()
btc['Upper'] = btc['MA20'] + 2*btc['STD20']
btc['Lower'] = btc['MA20'] - 2*btc['STD20']
btc['Daily Return']=btc['Close'].pct_change()
btc['MA7']=btc['Close'].rolling(7).mean()
btc['MA14']=btc['Close'].rolling(14).mean()
btc['MA30']=btc['Close'].rolling(30).mean()
btc['MA21']=btc['Close'].rolling(21).mean()
btc['OC_diff']=btc['Open']-btc['Close']
btc['range']=btc['High']-btc['Low']
#btc['OC_pct']=btc['Close'].pct_change()
btc['Volatility']=btc['Close'].rolling(7).std()
btc['Volatility2']=btc['Close'].rolling(30).std()
btc['RollingMax30'] = btc['Close'].rolling(30).max()
btc['HL_pct']=((btc['High']-btc['Low'])/btc['Close'])*100
btc['OC_pct']=((btc['Open']-btc['Close'])/btc['Close'])*100
btc['LogReturn'] = np.log(btc['Close'] / btc['Close'].shift(1))
btc['Year'] = btc['Date'].dt.year
btc['Month'] = btc['Date'].dt.month
btc['Quarter'] = btc['Date'].dt.quarter
btc['Weekday'] = btc['Date'].dt.dayofweek
btc['Day'] = btc['Date'].dt.day
btc['EMA20_EMA50_Ratio'] = btc['EMA20'] / btc['EMA50']
btc['Close_EMA20'] = btc['Close'] / btc['EMA20']
btc['ATR_Close'] = btc['ATR14'] / btc['Close']
btc['Volume_OBV'] = btc['Volume'] * btc['OBV']

btc=btc.dropna().reset_index(drop=True)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
x=btc[['OC_diff','OC_pct','range','Volatility','Volume','HL_pct','MA7','MA21','Open','Low','High']]
y=btc['Target']
scaler=StandardScaler()
x=scaler.fit_transform(x)
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
y_train.shape

(2011,)

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
x= btc[['Open',
    'High',
    'Low',
    'Close',
    'Volume',
    'EMA5',
    'EMA10',
    'EMA20',
    'EMA50',
    'EMA100',
    'EMA200',
    'SMA10',
    'SMA20',
    'SMA50',
    'SMA100',
    'SMA200',

    'WMA20',

    'HMA20','RSI14',

    'MACD_12_26_9',
    'MACDh_12_26_9',
    'MACDs_12_26_9',

    'ROC10',

    'MOM10',

    'CCI20',

    'STOCHk_14_3_3',
    'STOCHd_14_3_3',

    'WilliamsR'
]]
y=btc['Target']
print("BTC:", btc.shape)
print("X:", x.shape)
print("Y:", y.shape)
print("Missing values:\n", btc.isnull().sum())
print(x.index.equals(y.index))

print(x.isnull().sum().sum())

BTC: (2514, 83)
X: (2514, 28)
Y: (2514,)
Missing values:
 Date                 0
Open                 0
High                 0
Low                  0
Close                0
                    ..
Day                  0
EMA20_EMA50_Ratio    0
Close_EMA20          0
ATR_Close            0
Volume_OBV           0
Length: 83, dtype: int64
True
0


In [11]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x=scaler.fit_transform(x)

In [12]:
# Select features


# Select target
y = btc['Target']

print(x.shape)
print(y.shape)

# Split
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=False
)

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(2514, 28)
(2514,)
(2011, 28)
(503, 28)
(2011,)
(503,)


In [15]:
lr=LogisticRegression(C=0.01, max_iter=1000, solver='liblinear')
sv=SVC(C=10, degree=4, kernel='poly', probability=True)
xg=XGBClassifier(learning_rate= 0.01, max_depth= 3, n_estimators=50)
models=[lr,sv,xg]
for i in range(3):
    models[i].fit(x_train,y_train)
    print(f'{models[i]}:')
    print('Training Accuracy: ',metrics.roc_auc_score(y_train,models[i].predict_proba(x_train)[:,1]))
    print('Validation Accuracy: ',metrics.roc_auc_score(y_test,models[i].predict_proba(x_test)[:,1]))

LogisticRegression(C=0.01, max_iter=1000, solver='liblinear'):
Training Accuracy:  0.5583425656995139
Validation Accuracy:  0.5896200921951995
SVC(C=10, degree=4, kernel='poly', probability=True):
Training Accuracy:  0.6898314219840105
Validation Accuracy:  0.49607375615959304
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.01, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=50,
              n_jobs=No